In [19]:
from pathlib import Path

from datasets import load_dataset

# Zero-based worker/account ID and total number of accounts.
ID = 0
NUM_ACCOUNT = 4

if NUM_ACCOUNT <= 0:
    raise ValueError("NUM_ACCOUNT must be greater than 0")
if not 0 <= ID < NUM_ACCOUNT:
    raise ValueError("ID must be between 0 and NUM_ACCOUNT - 1")

OUTPUT_DIR = Path("output") / f"id_{ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "id.txt").write_text(f"{ID}\n", encoding="utf-8")

ds = load_dataset("AIGuruTinix/ViBioMIR", 'corpus')


/Users/buithianhdao/Library/Python/3.10/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
! pip install playwright
! playwright install


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 23.3 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [playwright]3 [playwright]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
182.1 MiB [                    ] 0% 303.1s182.1 MiB [                    ] 0% 277.8s182.1 MiB [                    ] 0% 427.6s182.1 MiB [                    ] 0% 325.2s182.1 MiB [                    ] 0% 294.9s182.1 MiB [                    ] 0% 864.2s182.1 MiB [                    ] 0% 724.2s182.1 MiB [                    ] 0% 991.4s182.1 MiB [                    ] 0% 878.6s182.1 MiB [                    ] 0% 789.7s182.1 MiB [                    ] 0% 728.9s182.1 MiB [                    ] 0% 681.5s182.1 MiB [                    ] 0% 678.5s182.1 MiB [                    ] 0% 635.9s182.1 MiB [                    ] 0% 611.0s182.1 MiB [                    ] 0% 585.4s182.1

In [ ]:
!playwright install-deps chromium


In [ ]:
!playwright install chromium


(node:8204) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
162.3 MiB [                    ] 0% 267.2s162.3 MiB [                    ] 0% 257.7s162.3 MiB [                    ] 0% 299.1s162.3 MiB [                    ] 0% 240.3s162.3 MiB [                    ] 0% 780.4s162.3 MiB [                    ] 0% 653.1s162.3 MiB [                    ] 0% 887.6s162.3 MiB [                    ] 0% 788.2s162.3 MiB [                    ] 0% 711.2s162.3 MiB [                    ] 0% 659.9s162.3 MiB [                    ] 0% 615.4s162.3 MiB [                    ] 0% 611.9s162.3 MiB [                    ] 0% 573.2s162.3 MiB [                    ] 0% 549.6s162.3 MiB [                    ] 0% 529.7s162.3 MiB [                    ] 0% 510.3s162.3 MiB [                 

In [ ]:
! pip install --upgrade trafilatura


  Using cached dateparser-1.4.3-py3-none-any.whl.metadata (25 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 20.4 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 18.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 22.4 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.6.3
    Uninstalling urllib3-2.6.3:
      Successfully uninstalled urllib3-2.6.3
  Attempting uninstall: lxml
    Found existing installation: lxml 6.0.4
    Uninstalling lxml-6.0.4:
      Successfully uninstalled lxml-6.0.4
  Attempting uninstall: charset_normalizer━━━━━━━━━━━━━━━━━━━━━━━━  3/12 [lxml]
    Found existing installation: charset-normalizer 3.4.1━━━━━  3/12 [lxml]
    Uninstalling charset-normalizer-3.4.1:━━━━━━━━━━━━━━━━━━━━  3/12 [lxml]
      Successfully uninstalled charset-normalizer-3.4.1━━━━━━━  3/12 [lxml]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [trafilatura] [trafilatur

In [17]:
import asyncio
import json
from urllib.parse import urlparse

import trafilatura
from playwright.async_api import (
    async_playwright,
    TimeoutError as PlaywrightTimeoutError,
    Error as PlaywrightError,
)


async def extract_url(url: str) -> dict:
    """
    Render URL bằng Playwright, sau đó dùng Trafilatura
    để extract main article + metadata.

    Có xử lý:
    - navigation chậm
    - redirect / page đang thay đổi DOM
    - JS client-side render
    - Trafilatura extract thất bại
    - nội dung article quá ngắn / rỗng
    """

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        page.set_default_timeout(15_000)

        try:
            # 1. Bắt đầu navigation.
            # Chỉ chờ đến commit để tránh bị kẹt bởi resource bên thứ ba.
            try:
                await page.goto(
                    url,
                    wait_until="commit",
                    timeout=30_000,
                )
            except PlaywrightTimeoutError:
                # Có thể page đã load được một phần,
                # nên vẫn thử xử lý tiếp.
                pass

            # 2. Chờ document cơ bản tồn tại.
            try:
                await page.wait_for_selector(
                    "body",
                    state="attached",
                    timeout=10_000,
                )
            except PlaywrightTimeoutError:
                pass

            # 3. Cho JS / redirect / client-side rendering chạy một chút.
            await asyncio.sleep(2)

            # 4. Thử chờ DOM parse xong.
            # Không bắt buộc, để tránh bị treo.
            try:
                await page.wait_for_load_state(
                    "domcontentloaded",
                    timeout=10_000,
                )
            except PlaywrightTimeoutError:
                pass

            # 5. Lấy HTML.
            # Một số site có thể đang redirect/navigation đúng lúc này,
            # nên retry vài lần.
            html = None
            last_error = None

            for _ in range(3):
                try:
                    html = await page.content()
                    break

                except PlaywrightError as e:
                    last_error = e
                    await asyncio.sleep(1)

            if html is None:
                raise RuntimeError(
                    f"Unable to retrieve stable page content: {last_error}"
                )

        except Exception as e:
            return {
                "url": url,
                "domain": urlparse(url).netloc,
                "success": False,
                "error": str(e),
            }

        finally:
            await browser.close()

    # 6. Extract article bằng Trafilatura.
    result = trafilatura.extract(
        html,
        url=url,
        output_format="json",
        with_metadata=True,
        include_comments=False,
        include_tables=True,
        favor_precision=True,
    )

    if result is None:
        return {
            "url": url,
            "html": html,
            "domain": urlparse(url).netloc,
            "success": False,
            "error": "Trafilatura could not extract content",
        }

    try:
        data = json.loads(result)
    except json.JSONDecodeError as e:
        return {
            "url": url,
            "domain": urlparse(url).netloc,
            "success": False,
            "error": f"Invalid Trafilatura JSON: {e}",
        }

    # 7. Validate nội dung article.
    text = (data.get("text") or "").strip()

    if len(text) < 0:
        return {
            "url": data.get("url") or url,
            "domain": urlparse(url).netloc,
            "source": data.get("sitename") or data.get("hostname"),
            "title": data.get("title"),
            "text_length": len(text),
            "success": False,
            "error": "Extracted article content is empty or too short",
        }

    # 8. Output chuẩn hóa.
    return {
        "url": data.get("url") or url,
        "domain": urlparse(url).netloc,
        "data": html,
        "source": data.get("sitename") or data.get("hostname"),
        "title": data.get("title"),
        "author": data.get("author"),
        "published_date": data.get("date"),
        "description": data.get("description"),
        "text": text,
        "success": True,
    }


In [ ]:
import pandas as pd
r2ai_lucraftmeds = pd.DataFrame(columns=["doc_id","url","domain","source","title","author","published_date","description","text","success"])


In [ ]:
r2ai_lucraftmeds


In [23]:
urls_full = ds["train"]["url"]
len(urls_full)


4394718

In [21]:
total_rows = len(ds["train"])
start = total_rows * ID // NUM_ACCOUNT
end = total_rows * (ID + 1) // NUM_ACCOUNT
urls = [ds["train"][i] for i in range(start, end)]

print(f"ID {ID}/{NUM_ACCOUNT - 1}: processing rows {start}:{end} ({len(urls)} rows)")


ID 0/3: processing rows 0:1098679 (1098679 rows)


In [ ]:
len(urls)


In [ ]:
urls[0] if urls else None


In [ ]:
error_urls = []


In [ ]:
from tqdm.auto import tqdm
import pandas as pd

rows = []

for item in tqdm(urls, desc=f"Crawling ID {ID}"):
    doc_id = item["id"]
    url = item["url"]

    article = await extract_url(url)
        
    if "error" in article:
        print(f"Error at {url}: {article['error']}")
        error_urls.append({"url": url, "doc_id": doc_id, "error": article["error"]})
        continue

    article["doc_id"] = doc_id
    rows.append(article)

r2ai_lucraftmeds = pd.DataFrame(
    rows,
    columns=["doc_id", "url", "domain", "source", "title", "author",
             "published_date", "description", "text", "success"],
)


In [ ]:
df_errors = pd.DataFrame(error_urls, columns=["doc_id", "url", "error"])
df_errors.to_csv(OUTPUT_DIR / "failed_urls.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(error_urls)} failed URLs to {OUTPUT_DIR / 'failed_urls.csv'}")


In [ ]:
urls[208] if len(urls) > 208 else None


In [ ]:
rows[0] if rows else None


In [ ]:
len(rows)


In [ ]:
len(rows[0]['text']) if rows else 0


In [ ]:
for i in range(len(rows)):
    if len(rows[i]['text']) < 1000 :
        break
        print(i)


In [ ]:
r2ai_lucraftmeds.to_csv(OUTPUT_DIR / "successful_urls.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(r2ai_lucraftmeds)} successful URLs to {OUTPUT_DIR / 'successful_urls.csv'}")
